# Customer Segmentation (Coursework 1) — Technical Implementation

This notebook implements an end-to-end segmentation pipeline for the 3,000-customer convenience retail dataset (6 months).

**Outputs created:**
- `customer_segments.csv` (customer_number → segment id)
- Summary tables used in the report

> Per coursework instruction, segments are named **Customer clutter 1..k**.


In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
# Paths (update if needed)
DATA_DIR = '.'

customers = pd.read_csv(f'{DATA_DIR}/customers_sample.csv')
category_spends = pd.read_csv(f'{DATA_DIR}/category_spends_sample.csv')
baskets = pd.read_csv(f'{DATA_DIR}/baskets_sample.csv')
lineitems = pd.read_csv(f'{DATA_DIR}/lineitems_sample.csv')

customers.shape, category_spends.shape, baskets.shape, lineitems.shape

((3000, 6), (3000, 21), (195547, 5), (1461315, 6))

In [3]:
def pounds_to_float(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    x = str(x).strip().replace('£', '').replace(',', '')
    try:
        return float(x)
    except:
        return np.nan

# Convert currency-like columns
for col in ['total_spend', 'average_spend']:
    customers[col] = customers[col].map(pounds_to_float)

for col in category_spends.columns:
    if col != 'customer_number':
        category_spends[col] = category_spends[col].map(pounds_to_float)

baskets['basket_spend'] = baskets['basket_spend'].map(pounds_to_float)
lineitems['spend'] = lineitems['spend'].map(pounds_to_float)

customers.head()

,customer_number,baskets,total_quantity,average_quantity,total_spend,average_spend
0,4749,220,260,1.181818,631.12,2.87
1,4757,248,333,1.342742,452.42,1.82
2,144,226,303,1.340708,261.16,1.16
3,572,285,346,1.214035,638.79,2.24
4,669,285,324,1.136842,561.42,1.97


## Data quality fix: Bakery spend

The provided `category_spends` table records bakery spend as a constant 0.  
As discussed in the coursework Q&A, we **recalculate bakery spend** from `lineitems_sample` and overwrite the bakery column.


In [4]:
# Recompute bakery spend at customer level
bakery_spend = (lineitems.loc[lineitems['category'].str.upper().eq('BAKERY')]
                .groupby('customer_number')['spend'].sum())

category_spends['bakery'] = category_spends['customer_number'].map(bakery_spend).fillna(0.0)

category_spends['bakery'].describe()

count    3000.000000
mean       38.210123
std        36.496115
min         0.000000
25%        14.635000
50%        29.270000
75%        50.050000
max       444.370000
Name: bakery, dtype: float64

## Feature engineering from basket logs

We create temporal and mission indicators per customer:
- recency (days since last visit)
- average basket spend, quantity, category diversity
- weekend share and daypart (morning / lunch / evening) shares
- average inter-visit gap (days)


In [5]:
baskets['purchase_time'] = pd.to_datetime(baskets['purchase_time'])
max_date = baskets['purchase_time'].max()

baskets['dow'] = baskets['purchase_time'].dt.dayofweek
baskets['hour'] = baskets['purchase_time'].dt.hour

baskets['is_weekend'] = baskets['dow'] >= 5
baskets['is_morning'] = baskets['hour'].between(6, 11)
baskets['is_lunch'] = baskets['hour'].between(12, 14)
baskets['is_evening'] = baskets['hour'].between(17, 21)

agg = baskets.groupby('customer_number').agg(
    first_purchase=('purchase_time', 'min'),
    last_purchase=('purchase_time', 'max'),
    n_visits=('purchase_time', 'count'),
    avg_basket_spend=('basket_spend', 'mean'),
    std_basket_spend=('basket_spend', 'std'),
    avg_basket_qty=('basket_quantity', 'mean'),
    avg_basket_cats=('basket_categories', 'mean'),
    weekend_share=('is_weekend', 'mean'),
    morning_share=('is_morning', 'mean'),
    lunch_share=('is_lunch', 'mean'),
    evening_share=('is_evening', 'mean'),
)
agg['recency_days'] = (max_date - agg['last_purchase']).dt.days

# Inter-visit gaps
b_sorted = baskets.sort_values(['customer_number', 'purchase_time'])
b_sorted['prev_time'] = b_sorted.groupby('customer_number')['purchase_time'].shift(1)
b_sorted['gap_days'] = (b_sorted['purchase_time'] - b_sorted['prev_time']).dt.total_seconds() / 86400
gap_stats = b_sorted.groupby('customer_number')['gap_days'].agg(['mean', 'std', 'median'])
gap_stats.columns = ['avg_gap_days', 'std_gap_days', 'median_gap_days']

agg = agg.join(gap_stats, how='left').reset_index()
agg.head()

,customer_number,first_purchase,last_purchase,n_visits,avg_basket_spend,std_basket_spend,avg_basket_qty,avg_basket_cats,weekend_share,morning_share,lunch_share,evening_share,recency_days,avg_gap_days,std_gap_days,median_gap_days
0,14,2007-03-03 15:37:00,2007-08-30 17:04:00,56,12.066429,8.701707,9.482143,4.464286,0.339286,0.071429,0.160714,0.160714,1,3.273826,2.305923,2.193750
1,45,2007-03-01 13:51:00,2007-08-30 16:59:00,33,17.749394,8.423640,19.848485,6.393939,0.000000,0.151515,0.636364,0.060606,1,5.691580,2.873620,6.705208
2,52,2007-03-07 17:12:00,2007-08-29 11:51:00,59,3.765763,2.914737,4.983051,2.949153,0.016949,0.355932,0.508475,0.033898,2,3.013398,2.811116,1.992361
3,61,2007-03-02 16:23:00,2007-08-28 16:25:00,37,14.807297,10.616203,13.486486,6.027027,0.135135,0.081081,0.216216,0.108108,3,4.972261,3.731403,4.126736
4,63,2007-03-02 12:38:00,2007-08-24 17:42:00,48,6.111250,5.434171,5.854167,3.666667,0.187500,0.187500,0.687500,0.083333,7,3.727896,2.564381,3.011806


## Combine customer-level tables

We merge:
- customer summary (`customers`)
- category spends (`category_spends`)
- engineered basket features (`agg`)


In [6]:
df = (customers
      .merge(category_spends, on='customer_number', how='left')
      .merge(agg, on='customer_number', how='left'))

df.shape, df.isna().mean().sort_values(ascending=False).head(10)

((3000, 41),
 std_gap_days        0.002
 median_gap_days     0.001
 std_basket_spend    0.001
 avg_gap_days        0.001
 seasonal_gifting    0.000
 discount_bakery     0.000
 practical_items     0.000
 first_purchase      0.000
 last_purchase       0.000
 n_visits            0.000
 dtype: float64)

## Category preference features (shares)

To reduce scale effects, we convert category spends into *shares* of total category spend.


In [7]:
category_cols = [c for c in category_spends.columns if c != 'customer_number']

df['cat_spend_total'] = df[category_cols].sum(axis=1)

for c in category_cols:
    df[f'share_{c}'] = df[c] / df['cat_spend_total'].replace(0, np.nan)

share_cols = [f'share_{c}' for c in category_cols]
df[share_cols] = df[share_cols].fillna(0)

df[share_cols].head()

,share_fruit_veg,share_dairy,share_confectionary,share_grocery_food,share_grocery_health_pets,share_bakery,share_newspapers_magazines,share_prepared_meals,share_soft_drinks,share_frozen,share_meat,share_tobacco,share_drinks,share_deli,share_world_foods,share_lottery,share_cashpoint,share_seasonal_gifting,share_discount_bakery,share_practical_items
0,0.022896,0.056424,0.038043,0.015829,0.051797,0.022785,0.033147,0.048438,0.027538,0.016146,0.003438,0.002662,0.197870,0.000000,0.000000,0.000000,0.459501,0.000000,0.000000,0.003486
1,0.181712,0.359798,0.101742,0.046771,0.046240,0.019009,0.003094,0.090403,0.041886,0.000000,0.013417,0.010300,0.048274,0.031365,0.000000,0.000000,0.000000,0.005990,0.000000,0.000000
2,0.135243,0.140412,0.187242,0.245597,0.030441,0.037180,0.017154,0.054794,0.001608,0.008003,0.032049,0.000000,0.009764,0.014550,0.081100,0.000000,0.000000,0.004863,0.000000,0.000000
3,0.062587,0.026347,0.068630,0.021102,0.005683,0.088715,0.066923,0.013760,0.015655,0.031074,0.032640,0.425555,0.000000,0.058032,0.017189,0.029744,0.000000,0.021776,0.009268,0.005323
4,0.026166,0.032792,0.011631,0.006181,0.041110,0.031848,0.033308,0.016316,0.066528,0.008870,0.084055,0.039756,0.569164,0.000000,0.001763,0.022265,0.000000,0.005290,0.000000,0.002957


## Clustering inputs

We use a compact set of behavioural + mission features:
- value + basket economics: total spend, average spend, spend per item, basket spend/qty
- engagement: visits, recency, inter-visit gap
- time pattern: weekend & daypart shares
- selected category preference shares

Skewed scale features are log(1+x) transformed; then all inputs are standardised.


In [8]:
df['spend_per_item'] = (df['total_spend'] / df['total_quantity'].replace(0, np.nan)).fillna(0)

feat_cols = [
    'total_quantity', 'average_quantity', 'total_spend', 'average_spend',
    'recency_days', 'n_visits',
    'avg_basket_spend', 'avg_basket_qty', 'avg_basket_cats',
    'weekend_share', 'morning_share', 'lunch_share', 'evening_share',
    'avg_gap_days', 'std_gap_days',
    # category preferences (shares)
    'share_tobacco', 'share_drinks', 'share_soft_drinks', 'share_prepared_meals',
    'share_cashpoint', 'share_lottery', 'share_confectionary', 'share_fruit_veg', 'share_dairy'
]

X = df[feat_cols].copy()

log_cols = ['total_quantity', 'total_spend', 'n_visits', 'avg_basket_spend', 'avg_basket_qty', 'avg_gap_days', 'std_gap_days']
for c in log_cols:
    X[c] = np.log1p(X[c].clip(lower=0))

X = X.fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X.shape

(3000, 24)

## Choosing k and fitting the final model

We explored k ∈ {5, 6, 7}. In retail data, silhouette scores are often modest; we therefore select k primarily on interpretability and business distinctness of missions.

Final choice: **k = 6**.


In [9]:
results = []
for k in [5, 6, 7]:
    km = KMeans(n_clusters=k, random_state=42, n_init=5, max_iter=200, algorithm='elkan')
    lab = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, lab, sample_size=800, random_state=42)
    results.append((k, km.inertia_, sil))
results

[(5, 52267.845229809565, 0.09645728673632757),
 (6, 50536.355100487264, 0.08223023047548296),
 (7, 48867.01146603342, 0.09584864708834981)]

In [10]:
k = 6
kmeans = KMeans(n_clusters=k, random_state=42, n_init=5, max_iter=200, algorithm='elkan')
labels = kmeans.fit_predict(X_scaled)

df['segment'] = labels + 1  # 1..k for reporting
df['segment'].value_counts().sort_index()

segment
1    304
2    268
3    375
4    872
5    514
6    667
Name: count, dtype: int64

## Segment summaries

We produce:
- size and core behavioural metrics per segment
- top category-share drivers per segment


In [11]:
summary_cols = [
    'n_visits','recency_days','total_spend','average_spend','total_quantity',
    'avg_basket_spend','avg_basket_qty','avg_basket_cats','weekend_share','evening_share'
]

seg_basic = (df.groupby('segment')
             .agg(customers=('customer_number','count'),
                  mean_total_spend=('total_spend','mean'),
                  mean_visits=('n_visits','mean'),
                  mean_recency_days=('recency_days','mean'),
                  mean_avg_basket_spend=('avg_basket_spend','mean'),
                  mean_avg_basket_qty=('avg_basket_qty','mean'),
                  mean_cat_diversity=('avg_basket_cats','mean'),
                  weekend_share=('weekend_share','mean'),
                  evening_share=('evening_share','mean'))
             .round(2))

seg_basic

,customers,mean_total_spend,mean_visits,mean_recency_days,mean_avg_basket_spend,mean_avg_basket_qty,mean_cat_diversity,weekend_share,evening_share
segment,,,,,,,,,
1,304,780.10,60.76,6.74,14.14,6.14,3.35,0.21,0.19
2,268,717.95,63.85,5.98,12.22,7.05,3.87,0.26,0.32
3,375,231.99,17.08,32.19,16.05,12.53,4.95,0.25,0.23
4,872,538.59,67.61,4.62,8.59,7.83,4.01,0.22,0.14
5,514,1302.91,132.36,1.73,10.45,7.73,4.16,0.22,0.18
6,667,978.01,39.84,5.58,26.91,21.83,6.79,0.24,0.17


In [12]:
# Top-5 category shares per segment
share_cols = [c for c in df.columns if c.startswith('share_')]
share_means = df.groupby('segment')[share_cols].mean()

topcats = {}
for seg in share_means.index:
    s = share_means.loc[seg].sort_values(ascending=False)
    topcats[seg] = list(s.head(5).index.str.replace('share_', ''))

topcats

{1: ['tobacco', 'cashpoint', 'dairy', 'drinks', 'lottery'],
 2: ['drinks', 'tobacco', 'meat', 'fruit_veg', 'dairy'],
 3: ['confectionary',
  'dairy',
  'fruit_veg',
  'grocery_health_pets',
  'grocery_food'],
 4: ['dairy',
  'fruit_veg',
  'grocery_food',
  'confectionary',
  'grocery_health_pets'],
 5: ['tobacco', 'cashpoint', 'dairy', 'grocery_health_pets', 'confectionary'],
 6: ['fruit_veg',
  'dairy',
  'grocery_health_pets',
  'grocery_food',
  'confectionary']}

## Export required results file: customer → segment

This is the deliverable `.csv` mapping customer IDs to segment id.


In [13]:
assign = df[['customer_number','segment']].sort_values('customer_number')
assign.to_csv('customer_segments.csv', index=False)
assign.head()

,customer_number,segment
38,14,4
54,45,6
61,52,4
63,61,6
60,63,4


In [15]:
df1=pd.read_csv('customer_segments.csv')

In [ ]:
df

In [16]:
df1['segment'].value_counts().sort_index()

segment
1    304
2    268
3    375
4    872
5    514
6    667
Name: count, dtype: int64